# CoreSight TrueUNet Colab Trainer 

### Step 1: Upload Data
Upload the `dataset.zip` file (generated by Antigravity) into the file explorer on the left side of Colab by dragging and dropping it.

### Step 2: Enable GPU
In the top menu, go to **Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4)**. Then click Save.

### Step 3: Unzip Data
Run the cell below to instantly extract your local dataset directly into the cloud server's active memory.

In [ ]:
!unzip -q dataset.zip -d /content/

### Step 4: Run the Training Loop
This cell contains your `TrueUNet` architecture, your `DiceBCELoss`, and your training pipeline precisely as it was running locally. Run it and watch it finish in 10-15 minutes instead of 3 hours, thanks to the T4 GPU!

In [ ]:
import os
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from pathlib import Path

def auto_crop_roi_paired(image_array, mask_array):
    gray = cv2.cvtColor(image_array, cv2.COLOR_RGB2GRAY)
    _, thresh = cv2.threshold(gray, 15, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: return image_array, mask_array
    largest_contour = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest_contour)
    return image_array[y:y+h, x:x+w], mask_array[y:y+h, x:x+w]

class UNetPancreasDataset(Dataset):
    def __init__(self, root_dir, is_train=True):
        self.root_dir = Path(root_dir)
        self.img_dir = self.root_dir / "data" / "images"
        self.mask_dir = self.root_dir / "data" / "masks"
        self.is_train = is_train
        self.files = [f.name for f in self.img_dir.iterdir() if f.is_file() and f.suffix == '.png']
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        fname = self.files[idx]
        image = cv2.imread(str(self.img_dir / fname))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(self.mask_dir / fname), cv2.IMREAD_GRAYSCALE)
        roi_img, roi_mask = auto_crop_roi_paired(image, mask)
        roi_img = cv2.resize(roi_img, (224, 224))
        roi_mask = cv2.resize(roi_mask, (224, 224), interpolation=cv2.INTER_NEAREST)
        pil_img, pil_mask = Image.fromarray(roi_img), Image.fromarray(roi_mask)
        if self.is_train and np.random.rand() > 0.5:
            pil_img, pil_mask = transforms.functional.hflip(pil_img), transforms.functional.hflip(pil_mask)
        img_tensor = transforms.ToTensor()(pil_img)
        img_tensor = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(img_tensor)
        mask_tensor = torch.from_numpy(np.array(pil_mask)).float() / 255.0
        return img_tensor, mask_tensor.unsqueeze(0)

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class TrueUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        self.up1 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv(256, 128)
        self.up3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv_up3 = DoubleConv(128, 64)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x = self.up1(x4)
        x = torch.cat([x, x3], dim=1)
        x = self.conv_up1(x)
        x = self.up2(x)
        x = torch.cat([x, x2], dim=1)
        x = self.conv_up2(x)
        x = self.up3(x)
        x = torch.cat([x, x1], dim=1)
        x = self.conv_up3(x)
        return self.outc(x)

class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1.0):
        super(DiceBCELoss, self).__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.smooth = smooth
    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)
        inputs = torch.sigmoid(inputs).view(-1)
        targets = targets.view(-1)
        intersection = (inputs * targets).sum()
        dice_loss = 1.0 - (2.0 * intersection + self.smooth) / (inputs.sum() + targets.sum() + self.smooth)
        return bce_loss + dice_loss

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Training on {device} 🚀')
    
    dataset = UNetPancreasDataset('/content')
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_set, val_set = random_split(dataset, [train_size, val_size])
    val_set.dataset.is_train = False
    
    # Using a larger batch size since we have a massive Free Cloud GPU!
    train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=32, shuffle=False)
    
    model = TrueUNet().to(device)
    criterion = DiceBCELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
    
    best_val_loss = float('inf')
    patience_counter = 0
    for epoch in range(30):
        model.train()
        running_loss = 0.0
        for inputs, masks in train_loader:
            inputs, masks = inputs.to(device), masks.to(device)
            optimizer.zero_grad()
            loss = criterion(model(inputs), masks)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, masks in val_loader:
                inputs, masks = inputs.to(device), masks.to(device)
                val_loss += criterion(model(inputs), masks).item() * inputs.size(0)
                
        t_loss = running_loss / train_size
        v_loss = val_loss / val_size
        scheduler.step(v_loss)
        print(f'Epoch {epoch+1}/30 | Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f}')
        
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            patience_counter = 0
            torch.save(model.state_dict(), '/content/true_unet_model.pth')
            print(' -> Saved Peak Accuracy Model! 🟢')
        else:
            patience_counter += 1
            if patience_counter >= 7:
                print('Early Stopping triggered - Maximum mathematical accuracy reached!')
                break

main()

### Step 5: Download the Output Engine
Once you see the `Early Stopping` message or the epochs finish, right click the **`true_unet_model.pth`** file in the left File Explorer panel and click **Download**.

Then drag that downloaded file straight into your local VS Code folder:
`c:\projects\CoreSight\pancrescan\unet_pipeline\true_unet_model.pth`